In [ ]:
import pandas as pd 
import numpy as np 
import re
from tqdm import tqdm
import matplotlib.pyplot as plt
from datetime import datetime

import pandas as pd
%run ./methods/methods.py

## Integration of biaises in reference dataset
#### 1. Load data


In [ ]:
date = datetime.today().strftime('%Y%m%d')

In [ ]:
data = pd.read_csv(f"./data/outputs/patients_geocoded_aligned_{date}_wstreetInfos.csv",sep=";").drop('Unnamed: 0',axis=1)

df_odonyme = pd.read_csv("./data/odonymes.txt",delimiter="|")
liste_odonyme = pd.Series( df_odonyme['terme'])


df_biaises = pd.read_csv('./data/biaises_identified.csv',sep="|",names=['index','biais'], index_col='index')
df_biaises = df_biaises.drop(index =[np.nan],axis=0)

odonyme = df_odonyme['terme'].str.upper()
odonyme = odonyme[~odonyme.duplicated()]
biais = df_biaises['biais'].str.upper()

#### 2. Position identification 

In [ ]:
df_pos_biais = find_pos_elem(data, 'adr_init',biais,'biais','contains_biais','pos_biais',drop_col_elem=False)


#### 3. Comparison of street and biais positon 

In [ ]:
df = df_pos_biais.copy()
df['comp_biais_street'] = "NULL"
df['diff_pos'] = None


## contains_street is True and contains_biais is True : 
mask = ((df['contains_street_type']==True) & (df['contains_biais']==True) &  (~df['not_matched_brute'].isna()))

df.loc[mask,'diff_pos'] = df.loc[mask,'pos_street_type'] - df.loc[mask,'pos_biais']

df.loc[mask,'comp_biais_street'] = df['comp_biais_street'].where(df['diff_pos'] > 0, 'BEFORE')
df.loc[mask,'comp_biais_street'] = df['comp_biais_street'].where(df['diff_pos'] < 0, 'AFTER')

## contains_street is False and contains_biais is True : 

mask2 = ((df['contains_street_type']==False) & (df['contains_biais']==True) &  (~df['not_matched_brute'].isna()))

df.loc[mask2,'comp_biais_street'] = "NO STREET"


In [ ]:
df_biaised = df[(df['contains_biais']==True) & (~df['not_matched_brute'].isna())]

#### 4. View position of the biaises compared to the street 

In [ ]:
df_pos_biaises = df_biaised.groupby(['biais','comp_biais_street']).size().reset_index(name="count")


result = df_pos_biaises.loc[df_pos_biaises.groupby('biais')['count'].idxmax()]
result = result.drop('count',axis=1)


#### 5. Load Reference dataFrame 

In [ ]:
df_ref = pd.read_csv('./data/reference/fr-en-adresse-et-geolocalisation-etablissements-premier-et-second-degre.csv',sep=";",dtype=str)


#### 6. Data preparation and biaises integration 
- only Metropolitan French adresses 
- textual data cleaning 

In [ ]:
mask_metrop = (
    df_ref['code_postal_uai'].str.startswith('97') | 
    df_ref['code_postal_uai'].str.startswith('98') |
    df_ref['code_postal_uai'].str.startswith('99'))
df_ref_metrop = df_ref[~mask_metrop] 
df_ref_metrop = df_ref_metrop.dropna(subset=['adresse_uai'])
df_ref_metrop['adresse_uai'] = df_ref_metrop['adresse_uai'].str.upper() 

df_sample = df_ref_metrop.sample(15000)[['numero_uai','adresse_uai','localite_acheminement_uai','code_postal_uai','coordonnee_x','coordonnee_y','latitude','longitude']]

In [ ]:
def biais_integration(df_ref,biais,pos,col_to_add_biaises):
    df_ref_biaised = df_ref.copy()
    if pos =='AFTER': 
        df_ref_biaised[col_to_add_biaises] =  df_ref_biaised[col_to_add_biaises] + f" {biais}"
    if pos =='BEFORE':
        df_ref_biaised[col_to_add_biaises] =   f"{biais} " + df_ref_biaised[col_to_add_biaises]
    return df_ref_biaised

In [ ]:
import os
dict_df_ref_biaised = {}
path = "./data/reference/ref_biaised/"
for i in range(len(result)):
    row_values = result.iloc[i].tolist()  # Convert row to pandas Series, then to list
    df_temp = biais_integration(df_sample,row_values[0],row_values[1],'adresse_uai')
    
    name_df = f'df_ref_{row_values[0]}'
    dict_df_ref_biaised[name_df] = df_temp
    if not os.path.exists(path):
        os.makedirs(path)
    df_temp.to_csv(f'./data/reference/ref_biaised/{name_df}.csv',sep=",")

In [ ]:
%run ./methods/chunkcsv.py


In [ ]:
dirpath_biaised = "./data/reference/ref_biaised/"
chunks_dir_path = "./data/reference/ref_chunked/"
chunks_geo_dir_path = "./data/reference/ref_chunked_geocoded/"
output_geocoded_dir_path = "./data/reference/ref_biaised_geocoded/"

list_dir = [dirpath_biaised,chunks_dir_path,chunks_geo_dir_path,output_geocoded_dir_path]
for dir in list_dir: 
    if not os.path.exists(dir):
        os.makedirs(dir) 

In [ ]:
files_biaised = [f for f in os.listdir(dirpath_biaised) if os.path.isfile(os.path.join(dirpath_biaised, f))]
 
for file in tqdm(files_biaised) :
    # if file == 'df_ref_APPT.csv':
    label = file.split('_')[2].split('.')[0]
    print(f"Biais {label} traité")
 
    df_to_geocode = pd.read_csv(os.path.join(dirpath_biaised,file),sep=",")
    df_to_geocode['label'] = label
    df_to_geocode = df_to_geocode.rename({'coordonnee_x':'x_L93_ref',
                                        'coordonnee_y':'y_L93_ref',
                                        'longitude':'x_WGS84_ref',
                                        'latitude':'y_WGS84_ref'},axis=1)
 
    process_dataframe_in_chunks2(df = df_to_geocode, label= label ,chunk_size=1000,
                                 chunks_dir_path = chunks_dir_path, chunks_geo_dir_path = chunks_geo_dir_path,
                                 columns = {"columns": ['adresse_uai', 'code_postal_uai', 'localite_acheminement_uai']})
 
    label_to_search = label +"_"
    files_chunked_geocoded_labeled = [f for f in os.listdir(chunks_geo_dir_path) if (os.path.isfile(os.path.join(chunks_geo_dir_path, f))) and (label_to_search in f)]
    # print(files_chunked_geocoded)
 
    i =0
    for file in files_chunked_geocoded_labeled :
        if label in file :
            df = pd.read_csv(os.path.join(chunks_geo_dir_path,file),sep=",")
            if i ==0 :  
                label_processed_dataframe = df
            else :
                label_processed_dataframe = pd.concat([label_processed_dataframe,df],axis=0)
            i+=1
 
    label_processed_dataframe_cut = label_processed_dataframe[['numero_uai', 'adresse_uai', 'localite_acheminement_uai','code_postal_uai', 'x_L93_ref', 'y_L93_ref','x_WGS84_ref','y_WGS84_ref',
 
                                                    'latitude','longitude','result_housenumber','result_name','result_postcode','result_city','result_label','label']]
 
    path = os.path.join(output_geocoded_dir_path,f"{label}_ref_biaised.csv")
    label_processed_dataframe_cut.to_csv(path,sep=";")
